# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal-141206/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.** The decision this serves is "which pages first" — not "is this page declining, yes/no" in isolation. An editor doesn't want 16,000 binary flags; they want an ordered list they can work down until their time runs out. Underneath the ranking there's a classification-shaped signal (declining vs. not), but the actual output that gets used is a **priority score per page**, sorted, with the top of the list mattering most and precision at the top mattering more than precision everywhere.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Honest answer: right now I only have a rule-defined proxy, not a true observed target.**

`is_declining_label` = `trend_direction == "down"`, and `trend_direction` is itself computed from `trend_pct`, a fixed threshold rule (last-30d vs. prev-30d impressions, >20%/<-20%/else). That means using this label as my target would mean my model learns to reproduce a threshold rule, not to predict a real future outcome — the exact trap `framing-ml-problems` warns against ("a label that comes from someone's rule means your model learns the rule, not the world").

The fix is available in the warehouse data, not the starter CSV: `fact_content_daily_performance` covers ~17 months per client, so I can define a genuinely **observed** target — e.g., did this page's clicks/sessions actually drop over a real future window, measured independently of the pre-computed `trend_direction` — using a mid-panel month as my feature cutoff and a later month as the outcome, with the final month (`_sample`) held out untouched as a sealed test set.

For this notebook, I'll use `is_declining_label` as a **placeholder proxy** to sketch the pipeline mechanics, and flag explicitly that it must be replaced with a true forward-observed target before any real claims get made.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("/workspaces/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Placeholder proxy target — NOT a true observed outcome, see note above
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df['is_declining_label'].value_counts(normalize=True))
print("\nNote: this proxy is rule-derived from trend_pct/trend_direction.")
print("Real target must come from fact_content_daily_performance, a later time window,")
print("independent of this pre-computed rule.")

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64

Note: this proxy is rule-derived from trend_pct/trend_direction.
Real target must come from fact_content_daily_performance, a later time window,
independent of this pre-computed rule.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K, with K set to a realistic weekly review capacity (e.g. top 20).**

Given the editor-hours cost structure from ML-02 — a false positive wastes scarce editor time, a false negative is quieter but compounds — precision at the top of the queue matters more than recall across the whole list. Precision@20 answers exactly the question that matters: "of the pages I'm telling the editor to look at this week, how many were actually worth it?" I'll also report the base rate (54.2% declining) alongside it, since a metric only means something in contrast to what a random or naive ranking would achieve.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Baseline reference point for later: naive/random precision@20 would hover near the base rate
base_rate = df['is_declining_label'].mean()
print(f"Base rate (naive precision@K if ranking were random): {base_rate:.1%}")
print("Target: precision@20 on the real ranking should meaningfully beat this.")

Base rate (naive precision@K if ranking were random): 54.2%
Target: precision@20 on the real ranking should meaningfully beat this.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), pseudonymized, belonging to one of 32 `client_id`s, with metrics aggregated over its trailing 90-day window. Confirmed below with a grain check and shape/column verification against the data dictionary.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Shape: {df.shape}")
print(f"Unique content_id: {df['content_id'].nunique()}")  # should equal row count if 1 row = 1 page
print(f"Unique client_id: {df['client_id'].nunique()}")

df.head(3)

Shape: (30000, 45)
Unique content_id: 30000
Unique client_id: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Two findings from the starter data (ML-02) already rule out a simple if-statement:

- **Visibility doesn't cleanly separate risk.** Decline rate among top_3/page_1 pages (51.6%) is nearly identical to the overall rate (54.2%) — a rule like "prioritize visible pages" wouldn't actually find the pages that need attention.
- **Staleness alone isn't monotonic.** Decline rate by `days_since_last_update` bucket goes 51% → 59% → 61% → 47% → 60% — not a clean climb. A single threshold rule ("flag anything stale >90 days") would misfire in both directions.

Both signals matter, but neither works alone, and the way they interact — staleness combined with content type, search volume, and position tier together — is exactly the "real but tangled" pattern `framing-ml-problems` describes as ML's actual use case. A hand-written rule can encode one or two of these; a model can weigh all of them together and be checked against a held-out outcome to see if it's actually doing better than a rule would.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick supporting cross-tab: does combining two signals separate the groups better than either alone?
cross = df.groupby(['position_tier', pd.cut(df['days_since_last_update'], [0,90,365,99999])])['is_declining_label'].agg(['mean','count'])
cross

mean  count
position_tier days_since_last_update                 
deep          (0, 90]                 0.339980   1003
              (90, 365]               0.357595    316
page_1        (0, 90]                 0.549136   8395
              (90, 365]               0.619842   3417
              (365, 99999]            1.000000      2
page_3_5      (0, 90]                 0.538284   4336
              (90, 365]               0.596419   2904
              (365, 99999]            0.500000      2
striking      (0, 90]                 0.596673   4929
              (90, 365]               0.636211   2375
top_3         (0, 90]                 0.175703   1992
              (90, 365]               0.637195    328
              (365, 99999]            0.000000      1

A single-signal rule was already shown to fail in ML-02: staleness alone, across all pages, is non-monotonic (51% → 59% → 61% → 47% → 60% by bucket) — an if-statement like "flag anything stale >90 days" would misfire.

But splitting staleness by `position_tier` reveals the real pattern was hiding, not absent:

| Tier | 0–90 days | 90–365 days |
|---|---|---|
| deep | 34.0% | 35.8% |
| page_1 | 54.9% | 62.0% |
| page_3_5 | 53.8% | 59.6% |
| striking | 59.7% | 63.6% |
| top_3 | 17.6% | 63.7% |

Within **every** position tier, decline rate rises with staleness — consistently. (365+ day buckets are excluded here; they hold 1–2 rows per group and aren't reliable evidence. The `top_3` jump is the sharpest but rests on a smaller sample, 328 rows, than `page_1`'s 3,417 — worth trusting less until validated further.)

This is the case ML is actually for: staleness alone is too noisy to act on, but staleness **combined with** position tier is a real, consistent signal. A rule-writer would need a separate hand-tuned threshold per tier (and probably per content type, search volume band, etc., stacking further) to capture what a model can learn directly by weighing signals jointly and checking the result against held-out data.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.